# Supplementary figures in Altair

Each section renders one supplementary figure from the existing analysis pipeline using the reusable functions in `src/plots_altair/`.

In [1]:
from pathlib import Path

if Path.cwd().stem == "notebooks":
    %cd ..

%load_ext autoreload
%autoreload 2

import tomllib

import altair as alt
import numpy as np
import polars as pl

from src.data.database_manager import DatabaseManager
from src.experiments.measurement.stimulus_generator import StimulusGenerator
from src.log_config import configure_logging
from src.plots.averages_over_stimulus_seeds import average_over_stimulus_seeds
from src.plots.correlations import (
    calculate_correlations_per_trial,
    calculate_participant_stats,
)
from src.plots_altair import (
    plot_correlation_heatmap,
    plot_grand_averages_grid,
    plot_participant_correlations,
    plot_stimulus_intervals,
    plot_stimulus_seed_grid,
    style_figure,
)

configure_logging(
    ignore_libs=("Comm", "bokeh", "tornado", "matplotlib"),
)
pl.Config.set_tbl_rows(12)
alt.data_transformers.disable_max_rows()

/Users/visser/drive/PhD/Code/pain-measurement


DataTransformerRegistry.enable('default')

## Shared stimulus data

In [2]:
configuration_path = Path("src/experiments/measurement/measurement_config.toml")
with configuration_path.open("rb") as file:
    stimulus_config = dict(tomllib.load(file)["stimulus"])

stimulus_config.update(
    {
        "temperature_baseline": 44.5,
        "temperature_range": 3,
    }
)
stimulus_seeds = stimulus_config["seeds"]
stimulus_133 = StimulusGenerator(stimulus_config, seed=133)

seed_grid_config = {**stimulus_config, "sample_rate": 2}
stimulus_frames = []
for seed in stimulus_seeds:
    generated = StimulusGenerator(seed_grid_config, seed=seed)
    stimulus_frames.append(
        pl.DataFrame(
            {
                "seed": [seed] * len(generated.y),
                "time_s": np.arange(len(generated.y)) / generated.sample_rate,
                "temperature": generated.y,
            }
        )
    )
stimuli = pl.concat(stimulus_frames)

## Step 1 — Stimulus interval types

In [3]:
stimulus_intervals_chart = plot_stimulus_intervals(
    stimulus_133,
    interval_order=None,
    interval_labels=None,
    interval_colors=None,
    width=900,
    height=390,
    bar_size=38,
    title=None,
)
style_figure(stimulus_intervals_chart)

alt.Chart(...)

## Step 2 — Temperature curves for all stimulus seeds

In [4]:
stimulus_seed_grid_chart = plot_stimulus_seed_grid(
    stimuli,
    columns=3,
    width=300,
    height=105,
    line_color="#17396b",
    line_width=2,
    panel_spacing=12,
    header_font_size=15,
    title=None,
)
style_figure(stimulus_seed_grid_chart)

alt.FacetChart(...)

## Shared exploratory-data averages

In [5]:
database = DatabaseManager()
with database:
    explore_data = database.get_trials("Explore_Data", exclude_problematic=True)

explore_data = explore_data.rename(
    {
        "rating": "pain_rating",
        "pupil": "pupil_diameter",
    }
)
physiological_signals = [
    "temperature",
    "pain_rating",
    "pupil_diameter",
    "eda_tonic",
    "eda_phasic",
    "heart_rate",
]
facial_signals = [
    "temperature",
    "brow_furrow",
    "cheek_raise",
    "mouth_open",
    "nose_wrinkle",
    "upper_lip_raise",
]
all_average_signals = list(
    dict.fromkeys([*physiological_signals, *facial_signals])
)
averages = average_over_stimulus_seeds(
    explore_data,
    all_average_signals,
    scaling="min_max",
    bin_size=0.1,
    confidence_level=0.95,
)

## Step 3 — Grand-averaged physiological signals

In [6]:
physiological_averages_chart = plot_grand_averages_grid(
    averages,
    physiological_signals,
    signal_labels=None,
    signal_colors=None,
    columns=3,
    width=330,
    height=145,
    display_step_ms=1000,
    y_domain=(-0.05, 1.05),
    line_width=1.5,
    line_opacity=0.95,
    show_ci=True,
    ci_opacity=0.13,
    column_spacing=18,
    row_spacing=14,
    legend_columns=6,
    panel_border_color="#606060",
    panel_border_width=0.8,
    title=None,
)
style_figure(physiological_averages_chart)

alt.VConcatChart(...)

## Step 4 — Grand-averaged facial expressions

In [7]:
facial_averages_chart = plot_grand_averages_grid(
    averages,
    facial_signals,
    signal_labels=None,
    signal_colors=None,
    columns=3,
    width=330,
    height=145,
    display_step_ms=1000,
    y_domain=(-0.05, 1.05),
    line_width=1.5,
    line_opacity=0.95,
    show_ci=True,
    ci_opacity=0.13,
    column_spacing=18,
    row_spacing=14,
    legend_columns=6,
    panel_border_color="#606060",
    panel_border_width=0.8,
    title=None,
)
style_figure(facial_averages_chart)

alt.VConcatChart(...)

## Step 5 — Participant-level correlations with temperature

In [8]:
correlation_data = explore_data.filter(
    pl.col("normalized_timestamp") >= 20 * 1000
)
physiological_targets = [
    "pain_rating",
    "pupil_diameter",
    "eda_tonic",
    "eda_phasic",
    "heart_rate",
]
physiological_trial_correlations = calculate_correlations_per_trial(
    correlation_data,
    "temperature",
    physiological_targets,
)
physiological_participant_stats = calculate_participant_stats(
    physiological_trial_correlations,
    physiological_targets,
)
physiological_correlations_chart = plot_participant_correlations(
    physiological_participant_stats,
    physiological_targets,
    reference="temperature",
    signal_labels=None,
    signal_colors=None,
    width=1200,
    height=440,
    y_domain=(-0.4, 1.0),
    point_size=58,
    point_opacity=0.72,
    error_bar_width=1.3,
    cap_size=8,
    zero_line_opacity=0.35,
    participant_group_padding=0.45,
    vertical_grid_opacity=0.35,
    legend_columns=5,
    title=None,
)
style_figure(physiological_correlations_chart)

alt.LayerChart(...)

In [9]:
facial_targets = [
    "brow_furrow",
    "cheek_raise",
    "mouth_open",
    "upper_lip_raise",
    "nose_wrinkle",
]
facial_trial_correlations = calculate_correlations_per_trial(
    correlation_data,
    "temperature",
    facial_targets,
)
facial_participant_stats = calculate_participant_stats(
    facial_trial_correlations,
    facial_targets,
)
facial_correlations_chart = plot_participant_correlations(
    facial_participant_stats,
    facial_targets,
    reference="temperature",
    signal_labels=None,
    signal_colors=None,
    width=1200,
    height=440,
    y_domain=(-0.7, 1.0),
    point_size=58,
    point_opacity=0.72,
    error_bar_width=1.3,
    cap_size=8,
    zero_line_opacity=0.35,
    participant_group_padding=0.45,
    vertical_grid_opacity=0.35,
    legend_columns=5,
    title=None,
)
style_figure(facial_correlations_chart)

alt.LayerChart(...)

## Step 6 — Facial-expression correlation matrix

In [10]:
facial_correlation_features = [
    "temperature",
    "pain_rating",
    "cheek_raise",
    "mouth_open",
    "upper_lip_raise",
    "nose_wrinkle",
    "brow_furrow",
]
facial_correlation_matrix_chart = plot_correlation_heatmap(
    averages,
    features=facial_correlation_features,
    skip_first_n_seconds=20,
    width=360,
    height=360,
    legend_title="Pearson correlation coefficient",
    title=None,
)
style_figure(facial_correlation_matrix_chart)

alt.LayerChart(...)